# Shared CV folds for every model

One fixed 5-fold split of the 1152 training rows, saved as `folds.csv` (`id` → `fold` 0–4). For fold *f*, the rows with `fold != f` are the training part and the rows with `fold == f` are the validation part.

Every model notebook (SVR, XGBoost, MLP, …) reads this file instead of calling `KFold` itself, so all out-of-fold predictions line up row by row for stacking, whatever order the data is loaded in.

Run this notebook once. Rerun it only if the training rows change (outlier removal), and then rerun every model notebook too.

In [ ]:
# Build the split on the kept training rows and save it

import pandas as pd
import numpy as np
from sklearn.model_selection import KFold

X = pd.read_csv("../X_train_knn.csv", index_col="id")          # the 1152 kept rows
y = pd.read_csv("../y_train.csv", index_col="id").loc[X.index, "y"]

# same split as used so far in the SVR notebook (KFold, shuffled, random_state=42, on this row order)
folds = pd.Series(-1, index=X.index, name="fold")
for f, (_, va) in enumerate(KFold(n_splits=5, shuffle=True, random_state=42).split(X)):
    folds.iloc[va] = f

assert (folds >= 0).all() and folds.index.is_unique
folds.to_csv("folds.csv")

# check: every fold has about the same size and the same age distribution
display(y.groupby(folds).agg(["count", "mean", "std", "min", "max"]).round(1))

,count,mean,std,min,max
fold,,,,,
0,231,70.4,9.6,42.0,95.0
1,231,70.0,9.2,45.0,93.0
2,230,69.7,9.1,48.0,95.0
3,230,70.0,9.3,45.0,97.0
4,230,69.9,9.8,42.0,95.0


### What every model notebook must do for the stacking

Paste this at the end of your model notebook. Replace `model` with your tuned model and `svr` with your model's name (`xgb`, `gp`, …).

```python
fold = pd.read_csv("../Stacking/folds.csv", index_col="id").loc[X.index, "fold"]
oof = pd.Series(np.nan, index=X.index, name="svr")

for f in range(5):
    train, val = fold != f, fold == f
    model.fit(X[train], y[train])
    oof[val] = model.predict(X[val])

print("out-of-fold R2:", r2_score(y, oof))
oof.to_csv("../Stacking/oof_svr.csv")

model.fit(X, y)   # retrain on all rows
pd.Series(model.predict(X_test), index=X_test.index, name="y").to_csv("submission_svr.csv")
```

This creates two files:
- `Stacking/oof_svr.csv`: a prediction for each of the 1152 training rows, each made by a model that never saw that row;
- `submission_svr.csv` in your model folder: a prediction for each of the 776 test rows, in `sample.csv` format (`id,y`), so it can also be uploaded as it is.

Then add one line for your model to `MODELS` in `stacking.ipynb` (cell 2).